# GSM8K Solver-Critic Pipeline



In [ ]:
from pathlib import Path, PurePosixPath
import os
import shutil
import sys
import zipfile

WORKING_DIR = Path('/kaggle/working')
INPUT_DIR = Path('/kaggle/input')
PROJECT_ROOT = WORKING_DIR / 'project'

if INPUT_DIR.exists() and not os.environ.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception as exc:
        raise RuntimeError('Add an enabled Kaggle Secret named HF_TOKEN for the gated Llama model.') from exc
print('HF token configured:', bool(os.environ.get('HF_TOKEN')))

def is_within(path: Path, parent: Path) -> bool:
    try:
        path.resolve().relative_to(parent.resolve())
    except ValueError:
        return False
    return True

def extract_safely(archive_path: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            if not is_within(destination / member.filename, destination):
                raise RuntimeError(f'Unsafe archive member: {member.filename}')
        archive.extractall(destination)

def is_project_archive(archive_path: Path) -> bool:
    with zipfile.ZipFile(archive_path) as archive:
        return any(PurePosixPath(name).as_posix().endswith('configs/config.yaml') for name in archive.namelist())

if INPUT_DIR.exists():
    if not (PROJECT_ROOT / 'configs' / 'config.yaml').exists():
        archives = [path for path in sorted(INPUT_DIR.rglob('*.zip')) if is_project_archive(path)]
        if len(archives) == 1:
            extract_safely(archives[0], PROJECT_ROOT)
        elif len(archives) > 1:
            raise RuntimeError('Multiple project ZIP archives were found under /kaggle/input/.')
        else:
            source_configs = sorted(INPUT_DIR.rglob('configs/config.yaml'))
            if len(source_configs) != 1:
                raise FileNotFoundError('Expected exactly one project configs/config.yaml under /kaggle/input/.')
            shutil.copytree(source_configs[0].parent.parent, PROJECT_ROOT, dirs_exist_ok=True)
    extracted_configs = sorted(PROJECT_ROOT.rglob('configs/config.yaml'))
    if not extracted_configs:
        raise FileNotFoundError('The writable project directory has no configs/config.yaml.')
    PROJECT_ROOT = extracted_configs[0].parent.parent
else:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    PROJECT_ROOT = next((path for path in candidates if (path / 'configs' / 'config.yaml').exists()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError('Run this notebook from the project checkout.')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')


In [ ]:
# Run only if Kaggle reports a missing or incompatible package, then restart the session once.
# %pip install -q --upgrade "transformers==5.0.0" accelerate bitsandbytes datasets pyyaml


In [ ]:

RUN_PIPELINE = False
RUN_DEBUG = False
DEBUG_SAMPLE_SIZE = 5
DEBUG_OUTPUT_ROOT = Path('/kaggle/working/debug_5_llama_v7')
PILOT_OUTPUT_ROOT = Path('/kaggle/working/project_v7_1000')
RUN_QUESTION_LIMIT = 50  # Change to 1000 to resume and complete the fixed cohort.
EVALUATION_ROOT = DEBUG_OUTPUT_ROOT if RUN_DEBUG else PILOT_OUTPUT_ROOT if RUN_PIPELINE else PROJECT_ROOT

if RUN_PIPELINE:
    from src.interact import run_pipeline

    artifacts = run_pipeline(
        config_path=PROJECT_ROOT / 'configs' / 'config.yaml',
        output_root=PILOT_OUTPUT_ROOT,
        question_limit=RUN_QUESTION_LIMIT,
    )
    print(artifacts)
elif RUN_DEBUG:
    from src.interact import run_debug_pipeline

    artifacts = run_debug_pipeline(
        config_path=PROJECT_ROOT / 'configs' / 'config.yaml',
        output_root=DEBUG_OUTPUT_ROOT,
        sample_size=DEBUG_SAMPLE_SIZE,
    )
    print(artifacts)
else:
    print('Skipping Pipeline: Evaluation will use existing JSONL outputs.')


## Feasibility Checkpoint Evaluation

This reads only saved JSONL interactions, loads GSM8K gold answers, and writes `data/processed/dataset.csv` plus `results/metrics.json`. It does not train a classifier or calculate later-stage metrics.

In [ ]:
from src.verify import format_checkpoint_summary, run_feasibility_evaluation

raw_input = EVALUATION_ROOT / 'data' / 'raw'
if not any(raw_input.glob('*.jsonl')):
    raw_input = EVALUATION_ROOT / 'pipeline_outputs.jsonl'
if not raw_input.exists():
    raise FileNotFoundError('No saved JSONL interactions found in data/raw or pipeline_outputs.jsonl.')

evaluation = run_feasibility_evaluation(
    config_path=PROJECT_ROOT / 'configs' / 'config.yaml',
    raw_dir=raw_input,
    output_root=EVALUATION_ROOT,
)
print(format_checkpoint_summary(evaluation['metrics']))
print('dataset.csv:', evaluation['dataset_path'])
print('metrics.json:', evaluation['metrics_path'])
